# Выравнивание Qwen3-4B-Instruct под простые объяснения

Полный пайплайн alignment для компактной модели: SFT -> DPO по стилю -> Reward Model -> DPO по качеству -> SimPO. Цель - модель, которая по умолчанию, без специальных промптов, объясняет сложные темы простым языком, не теряя фактической точности.

Ноутбук запускается сверху вниз в Colab на T4 GPU (QLoRA, 4-bit). Все seed зафиксированы (`seed = 42`), генерация greedy. Каждая из пяти задач заканчивается ячейкой, которая печатает свою метрику.

**Перед запуском:** Runtime -> Change runtime type -> T4 GPU.

In [1]:
!pip install -q -U transformers==4.57.1 peft==0.15.2 trl==0.19.0 accelerate==1.6.0 bitsandbytes==0.45.5 datasets==3.2.0 scikit-learn==1.7.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires 

In [2]:
import os, json, pickle
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from scipy.sparse import hstack

SEED = 42
set_seed(SEED)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

assert torch.cuda.is_available(), "GPU не подключён: Runtime -> Change runtime type -> T4 GPU, затем Restart session"


In [3]:
REPO_URL = "https://github.com/Vubni/test.git"
if not os.path.exists("data/kid_adult.jsonl"):
    !git clone $REPO_URL repo
    os.chdir("repo")

Cloning into 'repo'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 35 (delta 15), reused 28 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 1.48 MiB | 16.62 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [4]:
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

kid_adult = read_jsonl("data/kid_adult.jsonl")
public_test_style = read_jsonl("data/public_test_style.jsonl")
good_bad = read_jsonl("data/good_bad.jsonl")
public_test_quality = read_jsonl("data/public_test_quality.jsonl")

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
base_model = prepare_model_for_kbit_training(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [6]:
with open("metrics/style_clf.pkl", "rb") as f:
    style_clf = pickle.load(f)
word_vec, char_vec = style_clf["vecs"]
style_lr = style_clf["clf"]

def p_simple(texts):
    X = hstack([word_vec.transform(texts), char_vec.transform(texts)])
    return style_lr.predict_proba(X)[:, 1]

@torch.no_grad()
def generate(model, prompts, max_new_tokens=256):
    outs = []
    for i, p in enumerate(prompts):
        messages = [{"role": "user", "content": p}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        outs.append(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        if (i + 1) % 10 == 0:
            print(f"generated {i + 1}/{len(prompts)}")
    return outs


## Задача 1 - SFT: перенос стиля

Дообучаем базовую модель на `kid_adult` (target - простой ответ `kid`), чтобы стиль простых объяснений стал свойством модели, а не промпта. Метрика - средний `P_simple` ответов на `public_test_style` (классификатор `style_clf`), генерация greedy.

In [7]:
def to_text(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["kid"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

sft_dataset = Dataset.from_list(kid_adult).map(to_text, remove_columns=["prompt", "kid", "adult"])

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="adapters/sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=base_model,
    args=sft_config,
    train_dataset=sft_dataset,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("adapters/sft")

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.901300
40,1.284700
60,1.185800
80,1.156900
100,1.086300
120,0.931800
140,0.927400
160,0.934900
180,0.931500
200,0.823900


In [9]:
sft_model = trainer.model
sft_model.eval()

prompts = [d["prompt"] for d in public_test_style]
generations = generate(sft_model, prompts)
scores = p_simple(generations)
print("P_simple (SFT):", scores.mean())

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generated 10/50
generated 20/50
generated 30/50
generated 40/50
generated 50/50
P_simple (SFT): 0.9686945652904089


## Задача 2 - DPO по стилю: закрепление предпочтения

Поверх SFT сливаем адаптер в веса и прогоняем DPO на парах `kid_adult` (chosen - простой `kid`, rejected - сложный `adult`). SFT-модель служит неявной reference. Ожидаем рост `P_simple` относительно Задачи 1.

In [10]:
import gc

del trainer, sft_model
gc.collect()
torch.cuda.empty_cache()

In [11]:
bf16_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
sft_merged = PeftModel.from_pretrained(bf16_base, "adapters/sft").merge_and_unload()
sft_merged.save_pretrained("sft_merged")
tokenizer.save_pretrained("sft_merged")

del bf16_base, sft_merged
gc.collect()
torch.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [12]:
dpo_base = AutoModelForCausalLM.from_pretrained(
    "sft_merged", quantization_config=bnb_config, device_map="auto"
)
dpo_base = prepare_model_for_kbit_training(dpo_base)

dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
def build_prompt(question):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True
    )

dpo_dataset = Dataset.from_list([
    {"prompt": build_prompt(d["prompt"]), "chosen": d["kid"], "rejected": d["adult"]}
    for d in kid_adult
])

In [14]:
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    output_dir="adapters/dpo_style",
    beta=0.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    max_length=1024,
    max_prompt_length=512,
)

dpo_trainer = DPOTrainer(
    model=dpo_base,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.train()
dpo_trainer.save_model("adapters/dpo_style")

Extracting prompt in train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
20,0.509400
40,0.163200
60,0.045800
80,0.024600


In [15]:
dpo_model = dpo_trainer.model
dpo_model.eval()

generations = generate(dpo_model, prompts)
scores = p_simple(generations)
print("P_simple (DPO style):", scores.mean())

generated 10/50
generated 20/50
generated 30/50
generated 40/50
generated 50/50
P_simple (DPO style): 0.9867507935407538


## Задача 3 - Reward Model: оценщик качества

Обучаем reward-модель (голова классификации поверх Qwen3-4B, QLoRA) на `good_bad`: хороший ответ должен получать больший скор, чем плохой. Метрика - pairwise accuracy на `public_test_quality`: доля пар, где RM ставит `chosen` выше `rejected`.

In [16]:
del dpo_trainer, dpo_base, dpo_model
gc.collect()
torch.cuda.empty_cache()

In [17]:
from transformers import AutoModelForSequenceClassification

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

rm_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=1, quantization_config=bnb_config, device_map="auto"
)
rm_base.config.pad_token_id = tokenizer.pad_token_id
rm_base = prepare_model_for_kbit_training(rm_base)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
def rm_text(instruction, response):
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": response},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

rm_dataset = Dataset.from_list([
    {"chosen": rm_text(d["instruction"], d["chosen"]), "rejected": rm_text(d["instruction"], d["rejected"])}
    for d in good_bad
])

In [19]:
from trl import RewardConfig, RewardTrainer

rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
    task_type="SEQ_CLS",
)

reward_config = RewardConfig(
    output_dir="adapters/reward_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    max_length=1024,
)

reward_trainer = RewardTrainer(
    model=rm_base,
    args=reward_config,
    train_dataset=rm_dataset,
    processing_class=tokenizer,
    peft_config=rm_lora_config,
)
reward_trainer.train()
reward_trainer.save_model("adapters/reward_model")

Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2226 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,0.414600
40,0.026500
60,0.025600
80,0.037700
100,0.001500
120,0.075000
140,0.013900
160,0.000100
180,0.000900
200,0.064600


In [20]:
rm_model = reward_trainer.model
rm_model.eval()

@torch.no_grad()
def rm_score(texts):
    scores = []
    for t in texts:
        inputs = tokenizer(t, return_tensors="pt", truncation=True, max_length=1024).to(rm_model.device)
        scores.append(rm_model(**inputs).logits.item())
    return scores

correct = 0
for d in public_test_quality:
    s_chosen, s_rejected = rm_score([
        rm_text(d["prompt"], d["chosen"]),
        rm_text(d["prompt"], d["rejected"]),
    ])
    correct += s_chosen > s_rejected

print("Reward model pairwise accuracy:", correct / len(public_test_quality))

Reward model pairwise accuracy: 1.0


## Задача 4 - DPO по качеству

Поверх модели со стилем (SFT + DPO-style) прогоняем DPO на `good_bad` (chosen - точный ответ, rejected - правдоподобно ошибочный). Метрика - implicit-preference accuracy на `public_test_quality`: доля пар, где модель даёт `chosen` больший **средний logprob на токен** ответа (нормировка на длину обязательна). Без генерации, детерминированно.

In [21]:
del reward_trainer, rm_base, rm_model
gc.collect()
torch.cuda.empty_cache()

In [22]:
fp16_base = AutoModelForCausalLM.from_pretrained("sft_merged", torch_dtype=torch.float16, device_map="auto")
dpo_style_merged = PeftModel.from_pretrained(fp16_base, "adapters/dpo_style").merge_and_unload()
dpo_style_merged.save_pretrained("dpo_style_merged")
tokenizer.save_pretrained("dpo_style_merged")

del fp16_base, dpo_style_merged
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [23]:
quality_base = AutoModelForCausalLM.from_pretrained(
    "dpo_style_merged", quantization_config=bnb_config, device_map="auto"
)
quality_base = prepare_model_for_kbit_training(quality_base)

quality_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [24]:
quality_dataset = Dataset.from_list([
    {"prompt": build_prompt(d["instruction"]), "chosen": d["chosen"], "rejected": d["rejected"]}
    for d in good_bad
])

In [ ]:
dpo_quality_config = DPOConfig(
    output_dir="adapters/dpo_quality",
    beta=0.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    max_length=1024,
    max_prompt_length=512,
)

dpo_quality_trainer = DPOTrainer(
    model=quality_base,
    args=dpo_quality_config,
    train_dataset=quality_dataset,
    processing_class=tokenizer,
    peft_config=quality_lora_config,
)
dpo_quality_trainer.train()
dpo_quality_trainer.save_model("adapters/dpo_quality")

Extracting prompt in train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
20,0.607400
40,0.397000
60,0.270100
80,0.200900
100,0.213900
120,0.165900


In [ ]:
@torch.no_grad()
def avg_logprob_per_token(model, prompt, completion):
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids
    full_ids = tokenizer(prompt + completion, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    logits = model(full_ids).logits[0, :-1]
    log_probs = torch.log_softmax(logits.float(), dim=-1)
    target_ids = full_ids[0, 1:]
    token_logprobs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    completion_len = full_ids.shape[1] - prompt_ids.shape[1]
    return token_logprobs[-completion_len:].mean().item()

def implicit_preference_accuracy(model):
    correct = 0
    for d in public_test_quality:
        prompt = build_prompt(d["prompt"])
        lp_chosen = avg_logprob_per_token(model, prompt, d["chosen"])
        lp_rejected = avg_logprob_per_token(model, prompt, d["rejected"])
        correct += lp_chosen > lp_rejected
    return correct / len(public_test_quality)

dpo_quality_model = dpo_quality_trainer.model
dpo_quality_model.eval()
print("Implicit-preference accuracy (DPO quality):", implicit_preference_accuracy(dpo_quality_model))

## Задача 5 - SimPO: сравнение методов

Та же цель, что в Задаче 4, но без отдельной reference-модели (экономия памяти) - SimPO поверх той же стилевой модели на `good_bad`. Метрика та же - implicit-preference accuracy на `public_test_quality`; сравниваем с DPO, чтобы оценить, во что обходится отказ от reference.

In [ ]:
del dpo_quality_trainer, quality_base, dpo_quality_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
simpo_base = AutoModelForCausalLM.from_pretrained(
    "dpo_style_merged", quantization_config=bnb_config, device_map="auto"
)
simpo_base = prepare_model_for_kbit_training(simpo_base)

simpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

In [ ]:
from trl import CPOConfig, CPOTrainer

simpo_config = CPOConfig(
    output_dir="adapters/simpo",
    loss_type="simpo",
    cpo_alpha=0.0,
    simpo_gamma=0.5,
    beta=2.0,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    max_length=1024,
    max_prompt_length=512,
)

simpo_trainer = CPOTrainer(
    model=simpo_base,
    args=simpo_config,
    train_dataset=quality_dataset,
    processing_class=tokenizer,
    peft_config=simpo_lora_config,
)
simpo_trainer.train()
simpo_trainer.save_model("adapters/simpo")

In [ ]:
simpo_model = simpo_trainer.model
simpo_model.eval()
print("Implicit-preference accuracy (SimPO):", implicit_preference_accuracy(simpo_model))